In [1]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os


/home/mouad/anaconda3/envs/rag-assistant/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Extraction du texte du PDF
pdf_filename = os.listdir("../data")[0]
doc = fitz.open(f"../data/{pdf_filename}")

text = ""
for page in doc:
    text += page.get_text()
doc.close()

print(f"Longueur du texte extrait : {len(text)} caractères")
print("\n--- Aperçu ---")
print(text[:500])

Longueur du texte extrait : 91859 caractères

--- Aperçu ---
SOUKAINA BOUAROUROU
soukaina.bouarourou@fsr.um5.ac.ma
CLOUD COMPUTING ET VIRTUALISATION
ANNÉE UNIVERSITAIRE 
2025 - 2026
Master  Ingénierie Informatique et Sécurité Systèmes 
OBJECTIFS
Maîtriser les technologies de virtualisation et leurs applications
Comprendre et maîtriser les modèles de cloud computing
Savoir déployer et administrer des infrastructures virtualisées (hyperviseurs, conteneurs, 
stockage réseau).
Maîtriser les différents modèles de déploiement cloud (public, privé, hybride, comm


In [3]:
# 2. Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_text(text)

print(f"\nNombre de chunks : {len(chunks)}")
print("\n--- Chunk 1 ---")
print(chunks[0])
print("\n--- Chunk 2 ---")
print(chunks[1])


Nombre de chunks : 204

--- Chunk 1 ---
SOUKAINA BOUAROUROU
soukaina.bouarourou@fsr.um5.ac.ma
CLOUD COMPUTING ET VIRTUALISATION
ANNÉE UNIVERSITAIRE 
2025 - 2026
Master  Ingénierie Informatique et Sécurité Systèmes 
OBJECTIFS
Maîtriser les technologies de virtualisation et leurs applications
Comprendre et maîtriser les modèles de cloud computing
Savoir déployer et administrer des infrastructures virtualisées (hyperviseurs, conteneurs, 
stockage réseau).

--- Chunk 2 ---
stockage réseau).
Maîtriser les différents modèles de déploiement cloud (public, privé, hybride, communautaire).
SOUKAINA BOUAROUROU
soukaina.bouarourou@fsr.um5.ac.ma
Chp1 : VIRTUALISATION
ANNÉE UNIVERSITAIRE 
2025 - 2026
INTRODUCTION A LA VIRTUALISATION
une entreprise veut une architecture 3-tier : 
1 serveur web
1 serveur d'application
1 serveur de base de données
Comment déployer tout ça sur du VRAI matériel sans se ruiner ?
SOLUTION NAÏVE :
Acheter 3 serveurs physiques →15 000€


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2338.95it/s]


In [6]:
import chromadb

client_db = chromadb.PersistentClient(path="../vectorstore")
collection = client_db.get_or_create_collection(name="mon_cours")
ids = [f"chunk_{i}" for i in range(len(chunks))]

embeddings = model.encode(chunks).tolist()
collection.add(documents=chunks, embeddings=embeddings, ids=ids)

In [ ]:
question = "c'est quoi le chunking ?"  # ou une vraie question sur ton PDF

# Étape 1 : encoder la question EXACTEMENT comme   les chunks
question_embedding = model.encode([question]).tolist()  # liste car encode() attend une liste

# Étape 2 : query ChromaDB avec cet embedding
results = collection.query(
    query_embeddings=question_embedding,
    n_results=3  # top 3 chunks les plus proches
)

# Étape 3 : afficher les résultats
for i, doc in enumerate(results["documents"][0]):
    print(f"--- Résultat {i+1} ---")
    print(doc)
    print()

--- Résultat 1 ---
stockage de serveur informatique distants par l’ intermediaire d’un
réseau , généralement l’internet
•
Ces serveurs sont loués à la demande, le plus souvent par tranche d'utilisation selon des
critères techniques (puissance, bande passante, etc.) mais également au forfait. But :
avoir
accès
et
partager
diverses
ressources
informatiques
(espaces
de
stockage,
capacités de calculs et réseau etc. …), pouvoir y accéder de « partout », grâce à Internet.
Définition du Cloud Computing
•

--- Résultat 2 ---
entreprises, qui ont alors plus de disponibilité pour des activités à haute
valeur
ajoutée.
•
La maintenance, la sécurisation et les évolutions des services étant à la charge
exclusive du prestataire, dont c'est généralement le cœur de métier, celles-ci ont
tendance à être mieux réalisées et plus rapidement que lorsque sous la responsabilité
du client (principalement lorsque celui-ci n'est pas une organisation à vocation
informatique).
Avantages et inconvénients du cloud
